In [1]:
import pandas as pd
import numpy as np
import os, sys
import requests
import time

PROJECT_ROOT = os.path.join(os.getcwd(), '..', '..')#os.path.join(os.path.dirname(__file__), '..', '..')
sys.path.insert(1, PROJECT_ROOT) 
from prepare_data import ExchangeData
from message_manager import MessageManager
message = MessageManager()

class BinanceLoader(ExchangeData):

    def __init__(self, exchange, symbol_type, timezone='Asia/Taipei'):
        super().__init__(exchange, symbol_type)
        self.timezone = timezone


    def get_ts13(self, datetime:str,):
        ts13 = int(pd.to_datetime(datetime).tz_localize(self.timezone).timestamp()*1000)
        return ts13
  

    def fetch_funding_rate(
            self, 
            url, 
            symbol, 
            start_ts13,
            limit, 
            columns, 
            need_col = ["datetime", "funding_rate"],
            end_ts13 = None,
        ):

        """
        Mind
        - timezone = 'UTC'
        - usd: see https://binance-docs.github.io/apidocs/futures/en/#get-funding-rate-history
        - coin: see https://binance-docs.github.io/apidocs/delivery/en/#get-funding-rate-history-of-perpetual-futures
        """
        item = 'funding_rate'
        data_li = []
        params = {
            'symbol': symbol, 
            'limit':limit,
            'endTime': end_ts13,
        }
        while True:
            message.fetching(item, symbol, start_time=pd.to_datetime(start_ts13, unit='ms').tz_localize('UTC'))
            # timezone of binance is UTC+0
            params['startTime'] = start_ts13                        # update start_time
            data = requests.get(url, params=params).json()
            data_li.extend(data) 
            if len(data) < limit:
                break    
            start_ts13 = data[-1]['fundingTime'] + 1

        df = pd.DataFrame(data_li)
        df.columns = columns
        df = df[need_col].set_index(['datetime']).sort_index()
        df.index = pd.to_datetime(df.index, unit='ms').tz_localize('UTC').tz_convert(self.timezone)
        return df


    def update_funding_rate(self, start:str, symbol_li:list=None, end:str=None):
        """
        Update funding rate for the given symbols.

        Args:
        - start (str): Start datetime. (e.g., '2023-1-1 12:00:00+08:00')
        - symbol_li (list): List of symbols to download.
        - end (str): End datetime. (see start)

        Example:
            update_ohlcv('2021-1-1', ['BTCUSDT', 'ETHUSDT'])

        Mind
        - No symbol checking in symbol_li.
        - timezone = 'UTC' in binance.
        - update the data at the time of execution if no 'end'. 
            - ( all symbols with the same ending datetime )
        """
        item = 'funding_rate'
        save_dir = os.path.join(PROJECT_ROOT, 'data_base', self.exchange, self.symbol_type, 'funding_rate')
        os.makedirs(save_dir, exist_ok=True)

        url = self.get_end_point() + self.get_suffix_funding_rate()
        start_ts13 = self.get_ts13(start)
        end_ts13 = self.get_ts13(end) if end else int(time.time()*1000)
        limit = self.get_limit_funding_rate()
        columns = self.get_columns_funding_rate()

        if not symbol_li:
            url_exchange_info = self.get_end_point() + self.get_suffix_exchange_info()
            exchange_info = requests.get(url_exchange_info).json()
            symbol_li = [i['symbol'] for i in exchange_info['symbols'] if i['quoteAsset']=='USDT']

        for symbol in symbol_li:
            symbol_amount = len(symbol_li)
            message.updating(item, symbol_li.index(symbol)+1, symbol_amount)

            file_name = f"{symbol}_funding_rate.pkl"
            file_path = os.path.join(save_dir, file_name)
            
            # no file
            if not os.path.isfile(file_path):
                message.necessary_creating(item, symbol)
                df = self.fetch_funding_rate(
                    url = url,
                    symbol = symbol,
                    start_ts13 = start_ts13,
                    end_ts13 = end_ts13,
                    limit = limit,
                    columns = columns,
                )
                df.to_pickle(file_path)
                message.donload_completed(item, symbol, file_path)

            # file exists
            else:          
                df_ori = pd.read_pickle(file_path)
                start_ts13_ori = int(df_ori.index[0].timestamp()*1000)
                end_ts13_ori = int(df_ori.index[-1].timestamp()*1000)
                update_li = [df_ori]

                if (start_ts13 > start_ts13_ori) & (end_ts13 < end_ts13_ori):
                    message.unnecessary_update(item, symbol)
                    continue
                
                # head
                if start_ts13 < start_ts13_ori:
                    df_new_head = self.fetch_funding_rate(
                        url = url,
                        symbol = symbol,
                        start_ts13 = start_ts13,
                        end_ts13 = start_ts13_ori-1,
                        limit = limit,
                        columns = columns,
                    )
                    update_li.insert(0, df_new_head) 

                # tail
                if end_ts13 > end_ts13_ori:
                    df_new_tail = self.fetch_funding_rate(
                        url = url,
                        symbol = symbol,
                        start_ts13 = end_ts13_ori+1,
                        end_ts13 = end_ts13,
                        limit = limit,
                        columns = columns,
                    )
                    update_li.append(df_new_tail)

                df_updated = pd.concat(update_li)
                df_updated.to_pickle(file_path)
                message.update_completed(item, symbol)


    def fetch_ohlcv(
            self, 
            url, 
            symbol, 
            freq, 
            start_ts13,
            limit, 
            columns, 
            need_col = ["datetime", "open", "high", "low", "close", "volume"],
            end_ts13 = None,
        ):

        """
        Mind
        - timezone = 'UTC'
        - spot: see https://binance-docs.github.io/apidocs/spot/en/#kline-candlestick-data
        - usd: see https://binance-docs.github.io/apidocs/futures/en/#kline-candlestick-data
        - coin: see https://binance-docs.github.io/apidocs/delivery/en/#kline-candlestick-data
        """
        item = 'ohlcv'
        data_li = []
        params = {
            'symbol': symbol, 
            'interval': freq, 
            'limit':limit,
            'endTime': end_ts13,
        }
        while True:
            message.fetching(item, symbol, start_time=pd.to_datetime(start_ts13, unit='ms').tz_localize('UTC'))
            # timezone of binance is UTC+0
            params['startTime'] = start_ts13  # update start_time
            data = requests.get(url, params=params).json()
            data_li.extend(data) 
            if len(data) < limit:
                break    
            start_ts13 = data[-1][0] + 1

        df = pd.DataFrame(data_li, columns=columns)[need_col].set_index(['datetime']).sort_index()
        df.index = pd.to_datetime(df.index, unit='ms').tz_localize('UTC').tz_convert(self.timezone)
        return df


    def update_ohlcv(self, start:str, freq:str, symbol_li:list=None, end:str=None):
        """
        Update ohlcv for the given symbols.

        Args:
        - start (str): Start datetime. (e.g., '2023-1-1 12:00:00+08:00')
        - freq (str): Frequency of data (e.g., '30m', '1d', '1h'). 
        - symbol_li (list): List of symbols to download.
        - end (str): End datetime. (see start)

        Example:
            update_ohlcv('2021-1-1', '30m', ['BTCUSDT', 'ETHUSDT'])

        Mind
        - No symbol checking in symbol_li.
        - timezone = 'UTC' in binance.
        - update the data at the time of execution if no 'end'. 
            - ( all symbols with the same ending datetime )
        """
        item = 'ohlcv'
        save_dir = os.path.join(PROJECT_ROOT, 'data_base', self.exchange, self.symbol_type, freq)
        os.makedirs(save_dir, exist_ok=True)

        url = self.get_end_point() + self.get_suffix_kline()
        start_ts13 = self.get_ts13(start)
        end_ts13 = self.get_ts13(end) if end else int(time.time()*1000)
        limit = self.get_limit_kline()
        columns = self.get_columns_kline()

        if not symbol_li:
            url_exchange_info = self.get_end_point() + self.get_suffix_exchange_info()
            exchange_info = requests.get(url_exchange_info).json()
            symbol_li = [i['symbol'] for i in exchange_info['symbols'] if i['quoteAsset']=='USDT']

        for symbol in symbol_li:
            symbol_amount = len(symbol_li)
            message.updating(item, symbol_li.index(symbol)+1, symbol_amount)

            file_name = f"{symbol}_ohlcv.pkl"
            file_path = os.path.join(save_dir, file_name)
            
            # no file
            if not os.path.isfile(file_path):
                message.necessary_creating(item, symbol)
                df = self.fetch_ohlcv(
                    url = url,
                    symbol = symbol,
                    freq = freq,
                    start_ts13 = start_ts13,
                    end_ts13 = end_ts13,
                    limit = limit,
                    columns = columns,
                )
                df.to_pickle(file_path)
                message.donload_completed(item, symbol, file_path)

            # file exists
            else:          
                df_ori = pd.read_pickle(file_path)
                start_ts13_ori = int(df_ori.index[0].timestamp()*1000)
                end_ts13_ori = int(df_ori.index[-1].timestamp()*1000)
                update_li = [df_ori]

                if (start_ts13 > start_ts13_ori) & (end_ts13 < end_ts13_ori):
                    message.unnecessary_update(item, symbol)
                    continue
                
                # head
                if start_ts13 < start_ts13_ori:
                    df_new_head = self.fetch_ohlcv(
                        url = url,
                        symbol = symbol,
                        start_ts13 = start_ts13,
                        end_ts13 = start_ts13_ori-1,
                        freq = freq,
                        limit = limit,
                        columns = columns,
                    )
                    update_li.insert(0, df_new_head) 

                # tail
                if end_ts13 > end_ts13_ori:
                    df_new_tail = self.fetch_ohlcv(
                        url = url,
                        symbol = symbol,
                        start_ts13 = end_ts13_ori+1,
                        end_ts13 = end_ts13,
                        freq = freq,
                        limit = limit,
                        columns = columns,
                    )
                    update_li.append(df_new_tail)

                df_updated = pd.concat(update_li)
                df_updated.to_pickle(file_path)
                message.update_completed(item, symbol)



In [ ]:
save_dir = r'C:\Users\a1222\Collections\Trading_System\API_Connecter\data_base\binance\usd\15m'
df_ori = pd.read_pickle(os.path.join(save_dir, 'BTCUSDT_ohlcv.pkl'))
df_ori

In [4]:
bl = BinanceLoader(exchange='binance', symbol_type='usd', timezone='UTC')
# binance_loader.get_limit_kline()
# binance_loader.get_columns_kline()

In [8]:
symbol_li = ['BTCUSDT', 'ETHUSDT', 'XRPUSDT', 'BNBUSDT', 'SOLUSDT']

bl.update_ohlcv(
    start = '2022-1-1',
    end = '2022-1-8',
    freq = '15m',
    symbol_li = symbol_li
)

updating ohlcv of 1/5 symbol.
ohlcv of BTCUSDT is no need to update.
updating ohlcv of 2/5 symbol.
ohlcv of ETHUSDT is no need to update.
updating ohlcv of 3/5 symbol.
ohlcv of XRPUSDT is no need to update.
updating ohlcv of 4/5 symbol.
ohlcv of BNBUSDT is no need to update.
updating ohlcv of 5/5 symbol.
ohlcv of SOLUSDT is no need to update.


In [28]:
def fetch_funding_rate(
        self, 
        url, 
        symbol, 
        start_ts13,
        limit, 
        columns, 
        need_col = ["datetime", "funding_rate"],
        end_ts13 = None,
    ):

    """
    Mind
    - timezone = 'UTC'
    - usd: see https://binance-docs.github.io/apidocs/futures/en/#get-funding-rate-history
    - coin: see https://binance-docs.github.io/apidocs/delivery/en/#get-funding-rate-history-of-perpetual-futures
    """
    item = 'funding_rate'
    data_li = []
    params = {
        'symbol': symbol, 
        'limit':limit,
        'endTime': end_ts13,
    }
    while True:
        message.fetching(item, symbol, start_time=pd.to_datetime(start_ts13, unit='ms').tz_localize('UTC'))
        # timezone of binance is UTC+0
        params['startTime'] = start_ts13                        # update start_time
        data = requests.get(url, params=params).json()
        data_li.extend(data) 
        if len(data) < limit:
            break    
        start_ts13 = data[-1]['fundingTime'] + 1

    df = pd.DataFrame(data_li)
    df.columns = columns
    df = df[need_col].set_index(['datetime']).sort_index()
    df.index = pd.to_datetime(df.index, unit='ms').tz_localize('UTC').tz_convert(self.timezone)
    return df

In [7]:
symbol_li = ['BTCUSDT', 'ETHUSDT', 'XRPUSDT', 'BNBUSDT', 'SOLUSDT']

bl.update_funding_rate(
    start = '2022-1-1',
    end = '2022-1-10',
    symbol_li = symbol_li
)

updating funding_rate of 1/5 symbol.
funding_rate of BTCUSDT is no need to update.
updating funding_rate of 2/5 symbol.
funding_rate of ETHUSDT is no need to update.
updating funding_rate of 3/5 symbol.
funding_rate of XRPUSDT is no need to update.
updating funding_rate of 4/5 symbol.
funding_rate of BNBUSDT is no need to update.
updating funding_rate of 5/5 symbol.
funding_rate of SOLUSDT is no need to update.


In [13]:
pd.to_datetime(df_ori.index, unit='minute')

DatetimeIndex(['2021-01-01 00:00:00.002000+00:00',
               '2021-01-01 08:00:00.006000+00:00',
               '2021-01-01 16:00:00.003000+00:00',
                      '2021-01-02 00:00:00+00:00',
                      '2021-01-02 08:00:00+00:00',
               '2021-01-02 16:00:00.006000+00:00',
               '2021-01-03 00:00:00.018000+00:00',
                      '2021-01-03 08:00:00+00:00',
               '2021-01-03 16:00:00.002000+00:00',
                      '2021-01-04 00:00:00+00:00',
               ...
               '2024-01-20 08:00:00.003000+00:00',
                      '2024-01-20 16:00:00+00:00',
                      '2024-01-21 00:00:00+00:00',
                      '2024-01-21 08:00:00+00:00',
                      '2024-01-21 16:00:00+00:00',
                      '2024-01-22 00:00:00+00:00',
                      '2024-01-22 08:00:00+00:00',
                      '2024-01-22 16:00:00+00:00',
                      '2024-01-23 00:00:00+00:00',
            

In [8]:
save_dir = r'C:\Users\a1222\Collections\Trading_System\data_base\binance\usd\funding_rate'
df_ori = pd.read_pickle(os.path.join(save_dir, 'BTCUSDT_funding_rate.pkl'))
df_ori

,funding_rate
datetime,
2021-01-01 00:00:00.002000+00:00,0.00022753
2021-01-01 08:00:00.006000+00:00,0.00026336
2021-01-01 16:00:00.003000+00:00,0.00034457
2021-01-02 00:00:00+00:00,0.00010000
2021-01-02 08:00:00+00:00,0.00020151
...,...
2024-01-22 00:00:00+00:00,0.00010000
2024-01-22 08:00:00+00:00,0.00010000
2024-01-22 16:00:00+00:00,0.00010000


In [ ]:
requests.get?

Signature: requests.get(url, params=None, **kwargs)
Docstring:
Sends a GET request.

:param url: URL for the new :class:`Request` object.
:param params: (optional) Dictionary, list of tuples or bytes to send
    in the query string for the :class:`Request`.
:param \*\*kwargs: Optional arguments that ``request`` takes.
:return: :class:`Response <Response>` object
:rtype: requests.Response
File:      c:\users\hottari\anaconda3\envs\env3.9\lib\site-packages\requests\api.py
Type:      function

In [ ]:
start = int(pd.to_datetime('2024-1-16').timestamp()*1000)
end = int(pd.to_datetime('2024-1-17').timestamp()*1000)

In [ ]:
suffix = '/fapi/v1/klines'
url = end_point + suffix
params = {
    'symbol': 'BTCUSDT',
    'interval': '1T',
    'startTime': start, 
    'endTime': end,
    'limit': 1500
}

In [ ]:
response = requests.get(
    url, 
    params = params
)
#result = response.json()

In [ ]:
response.text

'<html>\r\n<head><title>404 Not Found</title></head>\r\n<body>\r\n<center><h1>404 Not Found</h1></center>\r\n<hr><center>nginx</center>\r\n</body>\r\n</html>\r\n'

In [ ]:
requests.get?

Signature: requests.get(url, params=None, **kwargs)
Docstring:
Sends a GET request.

:param url: URL for the new :class:`Request` object.
:param params: (optional) Dictionary, list of tuples or bytes to send
    in the query string for the :class:`Request`.
:param \*\*kwargs: Optional arguments that ``request`` takes.
:return: :class:`Response <Response>` object
:rtype: requests.Response
File:      c:\users\hottari\anaconda3\envs\env3.9\lib\site-packages\requests\api.py
Type:      function

In [ ]:
response

<Response [404]>